In [ ]:
import os, sys
# Run from project root so all relative paths and imports work
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from DAS import DAS, MulDAS
import glob

In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import cv2
from config import VIDEO_PATH, LABELING_CSV

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)

vehicle_label = pd.read_csv(LABELING_CSV)

ModuleNotFoundError: No module named 'config'

In [4]:
vehicle_label

,frame_start,frame_end,vehicle_type,make_model,lane_position,estimated_weight,color,note,note2
0,1873.0,1877.0,pickup,NaN,NaN,NaN,silver,NaN,NaN
1,1882.0,1898.0,truck,NaN,NaN,NaN,black,"ups, 4 axles","2:1887, 3:1896"
2,1932.0,1935.0,suv,NaN,NaN,NaN,white,NaN,NaN
3,1956.0,1959.0,sedan,NaN,NaN,NaN,white,NaN,NaN
4,2063.0,2066.0,suv,NaN,NaN,NaN,grey,NaN,NaN
5,2262.0,2264.0,suv,Tesla model Y,NaN,NaN,white,NaN,NaN
6,2363.0,2365.0,suv,NaN,NaN,NaN,white,NaN,NaN
7,2388.0,2391.0,suv,NaN,NaN,NaN,grey,NaN,NaN
8,2418.0,2421.0,sedan,NaN,NaN,NaN,white,NaN,NaN
9,2440.0,2443.0,suv,NaN,NaN,NaN,white,NaN,NaN


In [5]:
def add_frame_end_bridge(
    df: pd.DataFrame,
    estimate_speed: float,
    side_length_ft: float,
    bridge_length_m: float,
    fps_video: float,
    speed_unit: str = "mph",
    frame_col: str = "frame_start",
    out_col: str = "frame_end_bridge",
) -> pd.DataFrame:
    """
    Add a new column `frame_end_bridge` = frame when the vehicle leaves the far end of the bridge.

    Parameters
    ----------
    df : DataFrame with at least `frame_start`
    estimate_speed : float
        Estimated speed of vehicles (scalar for all rows).
    side_length_ft : float
        Distance from the joint to the near edge of the bridge (feet).
    bridge_length_m : float
        Length of the bridge (meters).
    fps_video : float
        Video frames per second.
    speed_unit : {"mph","kmh","mps"}
        Unit for estimate_speed.
    frame_col : str
        Column name containing frame indices for the joint crossing (start).
    out_col : str
        Output column name for the computed end frame at bridge exit.

    Returns
    -------
    DataFrame (same object) with `out_col` added.
    """
    if frame_col not in df.columns:
        raise ValueError(f"DataFrame must contain '{frame_col}'")

    # Convert speed to m/s
    unit = speed_unit.lower()
    if unit == "mph":
        speed_mps = estimate_speed * 0.44704
    elif unit == "kmh":
        speed_mps = estimate_speed * (1000/3600)
    elif unit in ("mps", "m/s"):
        speed_mps = estimate_speed
    else:
        raise ValueError("speed_unit must be one of {'mph','kmh','mps'}")

    if speed_mps <= 0:
        raise ValueError("estimate_speed must be > 0")

    # Total distance from joint to far end of bridge (meters)
    side_m = side_length_ft * 0.3048
    total_m = 2*side_m + bridge_length_m

    # Time to traverse from joint to far end (seconds), then to frames
    time_s = total_m / speed_mps
    delta_frames = time_s * fps_video

    # Vectorized compute; keep dtype as float for fractional frames
    df[out_col] = df[frame_col].astype(float) + float(delta_frames)
    return df

In [6]:
# Example parameters
fps = fps
side_length_ft = 20     # joint → near bridge edge
bridge_length_m = 35    # actual bridge length
estimate_speed_mph = 70  # rough traffic speed

vehicle_label_adjusted = add_frame_end_bridge(
    df=vehicle_label,                # your manual label DataFrame
    estimate_speed=estimate_speed_mph,
    side_length_ft=side_length_ft,
    bridge_length_m=bridge_length_m,
    fps_video=fps,
    speed_unit="mph",                # "kmh" or "mps" also supported
    frame_col="frame_start",
    out_col="frame_end_bridge"
)

In [ ]:
import os
from config import DAS_RECORDING_DIR

files = glob.glob(os.path.join(DAS_RECORDING_DIR, "*Newville*"))
muldas = MulDAS(files, np.arange(26, 60))
das_array = muldas.data
das_array_processed = das_array

In [8]:
import os
from pathlib import Path
from typing import Tuple, Optional, Dict, Callable, List
import numpy as np
import pandas as pd

# --- helpers (unchanged) ---
def _hms_to_seconds(hms: str) -> int:
    return sum(int(x) * m for x, m in zip(hms.split(':'), [3600, 60, 1]))

def _frame_to_das_seconds(
    frame: float,
    fps_video: float,
    frame_ref: int,
    sys_time_ref_hms: str,
    das_start_hms: str
) -> float:
    t_ref = _hms_to_seconds(sys_time_ref_hms)
    t_das0 = _hms_to_seconds(das_start_hms)
    t_frame_abs = t_ref + (frame - frame_ref) / fps_video
    return t_frame_abs - t_das0

# --- new helper to detect vehicle_type column ---
def _detect_vehicle_type_col(df: pd.DataFrame) -> Optional[str]:
    candidates = ["vehicle_type", "type", "class", "category"]
    for c in candidates:
        if c in df.columns:
            return c
    return None

# --- prepare function (extended) ---
def prepare_das_windows_and_labels(
    das_array: np.ndarray,
    labels_csv_or_df,
    window_length_s: float,
    stride_s: float,
    fs_das: int = 2000,
    fps_video: float = 29.98762733,
    frame_ref: int = 97,
    sys_time_ref_hms: str = "10:27:09",
    das_start_hms: str = "10:24:47",
    out_dir: str = "data",
    csv_out: str = "labels.csv",
    *,
    # NEW: how to name “many vehicles present”
    multi_token: str = "mixed",
    # NEW: what to put when no vehicle overlaps (None -> keep NaN)
    none_token: Optional[str] = None,
    # NEW: plug-in aggregators for future columns; dict[col] = callable(Series)->value
    # If you pass a key that isn't in labels, it's ignored.
    aggregators: Optional[Dict[str, Callable[[pd.Series], object]]] = None,
    # NEW: optional explicit list of columns to aggregate (if None, auto-detect for vehicle_type only)
    extra_label_cols: Optional[List[str]] = None,

    start_frame_col: str = "frame_start",
    end_frame_col: str = "frame_end_bridge", 
):
    """
    Converts manual frame labels to DAS windows + per-window labels.
    Adds a 'vehicle_type' column:
      - exactly one overlapping event -> that event's type
      - >1 overlapping events -> multi_token (default: 'mixed')
      - 0 events -> none_token (default: NaN)
    The 'aggregators' dict lets you add more per-window label fields later.
    """

    # Load labels
    if isinstance(labels_csv_or_df, (str, Path)):
        df_lbl = pd.read_csv(labels_csv_or_df)
    else:
        df_lbl = labels_csv_or_df.copy()

    # UPDATED: check for your chosen start/end columns
    required = {start_frame_col, end_frame_col}
    assert required.issubset(df_lbl.columns), \
        f"labels must contain columns: {required}"

    # Map frames -> DAS seconds (use chosen columns)
    df_lbl['start_s'] = df_lbl[start_frame_col].astype(float).apply(
        lambda f: _frame_to_das_seconds(f, fps_video, frame_ref, sys_time_ref_hms, das_start_hms)
    )
    df_lbl['end_s'] = df_lbl[end_frame_col].astype(float).apply(
        lambda f: _frame_to_das_seconds(f, fps_video, frame_ref, sys_time_ref_hms, das_start_hms)
    )
    df_lbl['start_s'], df_lbl['end_s'] = np.minimum(df_lbl['start_s'], df_lbl['end_s']), np.maximum(df_lbl['start_s'], df_lbl['end_s'])
    
    # Compute processing bounds (indices)
    t_min, t_max = df_lbl['start_s'].min(), df_lbl['end_s'].max()
    i_min = max(0, int(np.ceil(t_min * fs_das)))
    i_max = min(das_array.shape[1], int(np.floor(t_max * fs_das)))

    win_len = int(round(window_length_s * fs_das))
    stride = int(round(stride_s * fs_das))
    if win_len <= 0 or stride <= 0:
        raise ValueError("window_length_s and stride_s must be > 0")

    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    # For overlap tests
    events_np = df_lbl[['start_s', 'end_s']].to_numpy()

    # --- aggregator plumbing ---
    # Default: only vehicle_type logic unless user passes more.
    if aggregators is None:
        aggregators = {}

    # Determine which column to treat as vehicle_type
    vt_col = _detect_vehicle_type_col(df_lbl)
    want_vehicle_type = vt_col is not None

    # If user specified extra_label_cols, keep only those that exist
    if extra_label_cols is not None:
        extra_cols = [c for c in extra_label_cols if c in df_lbl.columns]
    else:
        # auto-include vehicle_type only for now
        extra_cols = [vt_col] if want_vehicle_type else []

    # Provide a default aggregator for vehicle_type if present and not overridden
    def _vehicle_type_aggregator(series: pd.Series) -> object:
        # series contains the overlapping rows' vehicle types
        unique_non_nan = [str(x) for x in series.dropna().unique()]
        if len(series) == 0:
            return none_token  # no overlaps
        if len(unique_non_nan) == 0:
            # All NAs in overlaps -> treat like "present but unknown"
            # You could return a special token; we'll keep NaN by default
            return none_token
        # Count *events*, not unique types: “exactly one event” means len(series)==1
        if len(series) == 1:
            return unique_non_nan[0]
        # multiple events
        return multi_token

    if want_vehicle_type and vt_col in extra_cols and vt_col not in aggregators:
        aggregators[vt_col] = _vehicle_type_aggregator

    # rows to write
    rows = []
    sample_id, start_idx = 0, i_min
    last_start = i_max - win_len

    # Precompute constants for frame back-map
    das0 = _hms_to_seconds(das_start_hms)
    sys_ref = _hms_to_seconds(sys_time_ref_hms)

    while start_idx <= last_start:
        end_idx = start_idx + win_len
        x = das_array[:, start_idx:end_idx]

        # Window in seconds
        Ws_s, We_s = start_idx / fs_das, end_idx / fs_das

        # Which labels overlap?
        e_start, e_end = events_np[:, 0], events_np[:, 1]
        overlap = np.minimum(We_s, e_end) - np.maximum(Ws_s, e_start)
        mask = overlap > 0
        cnt = int(mask.sum())

        # Frame back-map
        start_frame = int(np.floor(frame_ref + ((Ws_s + das0 - sys_ref) * fps_video)))
        end_frame = int(np.ceil (frame_ref + ((We_s + das0 - sys_ref) * fps_video)))

        # Save window
        fname = f"sample_{sample_id:06d}.npy"
        fpath = out_path / fname
        np.save(fpath, x.astype(np.float32))

        # Base row
        row = {
            "sample_id": f"{sample_id:06d}",
            "data_path": str(fpath.as_posix()),
            "count": cnt,
            "start_frame": start_frame,
            "end_frame": end_frame
        }

        # Apply aggregators for extra columns (future-proof)
        if cnt > 0 and len(extra_cols) > 0:
            overlapping = df_lbl.loc[mask, extra_cols]
            for col in extra_cols:
                agg_fn = aggregators.get(col, None)
                if agg_fn is None:
                    # Default: first non-NA (keeps it simple/override later)
                    val = overlapping[col].dropna()
                    row[col] = val.iloc[0] if len(val) else (none_token if none_token is not None else np.nan)
                else:
                    row[col] = agg_fn(overlapping[col])
        else:
            # No overlaps: fill with none_token/NaN for requested extra columns
            for col in extra_cols:
                row[col] = none_token if none_token is not None else np.nan

        rows.append(row)
        sample_id += 1
        start_idx += stride

    df_out = pd.DataFrame(rows)
    df_out.to_csv(out_path / csv_out, index=False)
    print(f"Saved {len(df_out)} samples to: {out_path.resolve()}")
    print(f"Label CSV: {(out_path / csv_out).resolve()}")
    return df_out, out_path


In [ ]:
df_out, out_dir = prepare_das_windows_and_labels(
    das_array=das_array_processed,
    labels_csv_or_df=vehicle_label_adjusted,
    window_length_s=5.0,
    stride_s=1.0,
    fs_das=2000,
    fps_video=fps,
    frame_ref=115,
    sys_time_ref_hms="10:27:09",
    das_start_hms="10:24:47",
    out_dir="data/raw",
    csv_out="../labels.csv",
    start_frame_col="frame_start",
    end_frame_col="frame_end_bridge",
    multi_token="mixed",
    none_token="background"
)

In [10]:
from preprocessing import *

In [11]:
pp = make_preprocess(f_lo=1, f_hi=20, order=5)
# load and process a sample
fs_das = 5000    # example sampling rate
sample = load_and_preprocess("data/sample_000003.npy", fs=fs_das, preprocess=pp)

print(sample.shape)

(34, 10000)
